In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

df = pd.read_csv("data/E2.csv")
matches = df[["HomeTeam", "AwayTeam", "FTHG", "FTAG"]].copy()

# league averages
avg_home_goals = matches["FTHG"].mean()
avg_away_goals = matches["FTAG"].mean()

# per-team home and away rates
home = matches.groupby("HomeTeam").agg(
    home_scored=("FTHG", "mean"),
    home_conceded=("FTAG", "mean")
)
away = matches.groupby("AwayTeam").agg(
    away_scored=("FTAG", "mean"),
    away_conceded=("FTHG", "mean")
)
strength = home.join(away)
strength.head()

,home_scored,home_conceded,away_scored,away_conceded
HomeTeam,,,,
Barnsley,1.434783,1.565217,1.565217,1.608696
Birmingham,2.043478,0.478261,1.608696,0.869565
Blackpool,1.521739,1.260870,1.608696,1.347826
Bolton,1.565217,1.391304,1.347826,1.652174
Bristol Rvs,1.347826,1.434783,0.565217,1.869565


In [3]:
def expected_goals(home_team, away_team):
    # home team's expected goals: their home attack x away team's away defence, relative to league
    home_exp = (strength.loc[home_team, "home_scored"]
                * strength.loc[away_team, "away_conceded"]
                / avg_home_goals)
    # away team's expected goals: their away attack x home team's home defence, relative to league
    away_exp = (strength.loc[away_team, "away_scored"]
                * strength.loc[home_team, "home_conceded"]
                / avg_away_goals)
    return home_exp, away_exp

# test it on a strong home team vs a weaker side — adjust names to teams in YOUR data
expected_goals("Birmingham", "Burton")

(2.025021037868163, 0.3344891971192318)

In [5]:
home_exp, away_exp = expected_goals("Birmingham", "Burton")

# probability each side scores exactly 0, 1, 2, 3, 4, 5 goals
goals = range(0, 6)
home_probs = [poisson.pmf(g, home_exp) for g in goals]
away_probs = [poisson.pmf(g, away_exp) for g in goals]

for g in goals:
    print(f"{g} goals -> Birmingham {home_probs[g]:.3f}, Burton {away_probs[g]:.3f}")

0 goals -> Birmingham 0.132, Burton 0.716
1 goals -> Birmingham 0.267, Burton 0.239
2 goals -> Birmingham 0.271, Burton 0.040
3 goals -> Birmingham 0.183, Burton 0.004
4 goals -> Birmingham 0.092, Burton 0.000
5 goals -> Birmingham 0.037, Burton 0.000
